# MCP Server on Modal

This notebook does these steps:

1. Write an MCP server. The server uses the Chroma Cloud collection from the **Document Ingestion** notebook.
2. Do a test of the server on this computer.
3. Deploy the server to Modal.
4. Do a test of the deployed server.
5. Connect the server to Claude.

The server has one tool, `search_documents`. The tool gives only the text of the chunks, as `retrieve_text` in the ingestion notebook.

**Before you start:** Run the **Document Ingestion** notebook. Then the collection contains the chunks.

**Code style:** The code follows the same ASD-STE100 rules as the ingestion notebook.

## 1. Install the packages

Do this step one time only.

In [1]:
%pip install -q modal "mcp==2.3.0" "chromadb==1.5.9" python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Log in to Modal

Do this step one time only. The command opens a browser window. Log in, then come back to the notebook.

If you already have a Modal token on this computer, do not do this step.

In [2]:
import sys

# Remove the comment sign to log in.
# !{sys.executable} -m modal setup

## 3. Write the MCP server

The cell below writes the file `mcp_server.py`. The file has these parts:

- **The image.** The image has `chromadb` and `mcp`. The image build downloads the embedding model one time. Thus, a new container starts quickly.
- **The retrieval.** The functions are the same as in the ingestion notebook. The server uses the Chroma default embedding model (all-MiniLM-L6-v2), as the ingestion notebook does. Thus, the query vectors and the chunk vectors are the same kind.
- **The tool.** `search_documents(query, top_k)` gives a list of text chunks.
- **The web endpoint.** Modal serves the MCP server with the Streamable HTTP transport at the path `/mcp`.

The server gets the Chroma values from the `.env` file. Modal sends them to the container as a secret. The values do not go into the code.

The server is stateless. Thus, Modal can use more than one container, and each request can go to a different container.

In [3]:
%%writefile mcp_server.py
"""MCP server for the workshop documents. It runs on Modal.

The server has one tool: search_documents.
The tool finds the chunks that match a query in the Chroma Cloud collection.
The tool gives only the text of the chunks.
"""

import modal

APP_NAME = "workshop-documents-mcp"

# Use the same collection as the "Document Ingestion" notebook.
# The collection uses the Chroma default embedding model (all-MiniLM-L6-v2).
COLLECTION_NAME = "workshop_documents_default"
TOP_K = 3              # Number of chunks that one query gives.
MAX_TOP_K = 10         # Largest number of chunks that one query can give.


def download_model():
    """Download the default embedding model into the image."""
    from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

    DefaultEmbeddingFunction()(["Download the model."])


image = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_pip_install("chromadb==1.5.9", "mcp==2.3.0")
    .run_function(download_model)
)

app = modal.App(APP_NAME)

with image.imports():
    import os

    import chromadb
    from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
    from mcp.server.mcpserver import MCPServer
    from mcp.server.transport_security import TransportSecuritySettings


def connect_to_chroma():
    """Connect to Chroma Cloud. Return the client."""
    return chromadb.CloudClient(
        tenant=os.environ["CHROMA_TENANT"],
        database=os.environ["CHROMA_DATABASE"],
        api_key=os.environ["CHROMA_API_KEY"],
    )


def open_collection():
    """Open the collection that the ingestion notebook made."""
    return connect_to_chroma().get_collection(
        name=COLLECTION_NAME,
        embedding_function=DefaultEmbeddingFunction(),
    )


def retrieve_text(collection, query, top_k=TOP_K):
    """Find the chunks that match the query. Return only their text."""
    result = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents"],
    )
    return result["documents"][0]


def make_server():
    """Make the MCP server and its tool."""
    collection = open_collection()
    server = MCPServer(
        name="Workshop Documents",
        instructions=(
            "Use search_documents to find text in the workshop policy documents. "
            "Give the answer only from the text that the tool gives."
        ),
    )

    @server.tool()
    def search_documents(query: str, top_k: int = TOP_K) -> list[str]:
        """Find the parts of the workshop documents that match the query.

        Args:
            query: The question or the words to find.
            top_k: The number of text chunks to give (1 to 10).
        """
        top_k = max(1, min(top_k, MAX_TOP_K))
        return retrieve_text(collection, query, top_k)

    return server


@app.function(
    image=image,
    secrets=[modal.Secret.from_dotenv(__file__)],
    scaledown_window=300,
)
@modal.concurrent(max_inputs=20)
@modal.asgi_app()
def serve():
    """Serve the MCP server with the Streamable HTTP transport at the path /mcp."""
    return make_server().streamable_http_app(
        stateless_http=True,
        json_response=True,
        # The Modal URL is not localhost. Thus, turn off the localhost-only host check.
        transport_security=TransportSecuritySettings(enable_dns_rebinding_protection=False),
    )

Overwriting mcp_server.py


## 4. Do a test of the server on this computer

The test does not use the network for MCP. The client connects to the server object in this Python process.
The tool still gets the chunks from Chroma Cloud.

In [4]:
from pathlib import Path

from dotenv import load_dotenv
from mcp import Client

import mcp_server

load_dotenv(Path(".env"))

TEST_QUERY = "What are the principles of the IT architecture framework?"


def show_chunks(result):
    """Show the start of each text chunk in the tool result."""
    for rank, content in enumerate(result.content, start=1):
        print(f"  {rank}. {content.text[:300]}...")


async with Client(mcp_server.make_server()) as client:
    tools = await client.list_tools()
    print(f"Tools: {[tool.name for tool in tools.tools]}")
    result = await client.call_tool("search_documents", {"query": TEST_QUERY})

assert not result.is_error, "The tool gave an error."
print(f"QUERY: {TEST_QUERY}")
show_chunks(result)

/Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/pydantic/plugin/_schema_validator.py:40: UserWarning: ModuleNotFoundError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ModuleNotFoundError("No module named 'opentelemetry.exporter.otlp.proto.common._exporter_metrics'")
  plugins = get_plugins()


Tools: ['search_documents']


[10/21/26 05:07:54] INFO     HTTP Request: POST                                                     ]8;id=11769929;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=11769930;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://api.trychroma.com/api/v2/tenants/5514afd6-36af-4a72-be70-9f146                
                             638ea4e/databases/llm-demo/collections/b0f72d1e-0772-472e-a530-58fad6c                
                             13fd5/query "HTTP/1.1 200 OK"                                                         

QUERY: What are the principles of the IT architecture framework?
  1. 3. INFORMATION TECHNOLOGY ARCHITECTURE FRAMEWORK 16 The Ministry of Economy and Finance Information Technology Architecture Framework 2)- Process: Define the structure of the development, operation, and management of IT systems to change how IT systems work and how they can work together. Furthermor...
  2. 3. INFORMATION TECHNOLOGY ARCHITECTURE FRAMEWORK 23 The Ministry of Economy and Finance Information Technology Architecture Framework (B) Interoperability Framework: refers to the mechanism that allows two or more IT systems to interop with each other to share data or use data from each other. Lack ...
  3. once-only principles, interoperability as an ecosystem principle, and land and expand for efficiency principle. In addition to these three principles, the framework adds six key common principles for implementation, including (1)-Ownership, (2)-Interoperability, (3)-Development Standards, (4)-Open D...


## 5. Deploy the server to Modal

The first deployment builds the image. This can take some minutes. The next deployments use the same image if you do not change it.

Run this cell again after you change `mcp_server.py`.

In [5]:
!{sys.executable} -m modal deploy mcp_server.py

/Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/pydantic/plugin/_schema_validator.py:40: UserWarning: ModuleNotFoundError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ModuleNotFoundError("No module named 'opentelemetry.exporter.otlp.proto.common._exporter_metrics'")
  plugins = get_plugins()
⠦ Creating objects.....
⠏ Creating objects...
└── ⠸ Creating mount /Users/rinabuoy/Desktop/Workshop/Workshop 2/mcp_server.py: 
⠹ Creating objects...
├── ✓ Uploaded /Users/rinabuoy/Desktop/Workshop/Workshop 2/mcp_server.py
⠴ Creating objects..._model...
├── ✓ Uploaded /Users/rinabuoy/Desktop/Workshop/Workshop 2/mcp_server.py
⠇ Creating objects..._model...
├── ✓ Uploaded /Users/rinabuoy/Desktop/Workshop/Workshop 2/mcp_server.py
⠹ Creating objects...load_model.
├── ✓ Uploaded /Users/rinabuoy/Desktop/Workshop/Workshop 2/mcp_server.py
├── ✓ Created Function download_model.
⠼ Creating objects...
├── ✓ Uploaded /Users/rinabuoy/Desktop/Workshop/W

## 6. Get the URL of the server

The MCP endpoint is the web URL of the Modal function plus the path `/mcp`.

In [6]:
import modal

MCP_URL = modal.Function.from_name(mcp_server.APP_NAME, "serve").get_web_url() + "/mcp"
print(f"MCP URL: {MCP_URL}")

MCP URL: https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp


## 7. Do a test of the deployed server

The client connects to the server on Modal. The first request can be slow, because Modal must start a container.

In [7]:
TEST_QUERIES = [
    "What are the objectives of the financial technology development policy?",
    "What are the goals of the e-services for business strategy?",
    "What are the principles of the IT architecture framework?",
]

async with Client(MCP_URL) as client:
    tools = await client.list_tools()
    print(f"Tools: {[tool.name for tool in tools.tools]}")
    print()
    for query in TEST_QUERIES:
        result = await client.call_tool("search_documents", {"query": query, "top_k": 3})
        assert not result.is_error, f"The tool gave an error: {query}"
        assert result.content, f"The query gave no text: {query}"
        print(f"QUERY: {query}")
        show_chunks(result)
        print()

[10/21/26 05:09:22] INFO     HTTP Request: POST                                                     ]8;id=11769937;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=11769938;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py#1923\1923]8;;\
                             https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp                
                             "HTTP/1.1 200 OK"                                                                     

[10/21/26 05:09:23] INFO     HTTP Request: POST                                                     ]8;id=11769943;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=11769944;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py#1923\1923]8;;\
                             https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp                
                             "HTTP/1.1 200 OK"                                                                     

Tools: ['search_documents']



[10/21/26 05:09:24] INFO     HTTP Request: POST                                                     ]8;id=11769949;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=11769950;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py#1923\1923]8;;\
                             https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp                
                             "HTTP/1.1 200 OK"                                                                     

QUERY: What are the objectives of the financial technology development policy?
  1. tation, and (3)-further innovation based on the concept of “maximizing the benefits of the digital transformation in the financial sector and mitigating risks to a minimum level”. A total of 52 policy measures are set out and subdivided into individual goals and are to be achieved in five years. Amo...
  2. ement of financial inclusion, (2)-the maintenance of the financial sector stability, and (3)-the promotion of financial innovation. This policy aims to “develop a vibrant FinTech ecosystem through the development of policy enablers, the development of digital enablers, the promotion of the use and d...
  3. 13 3. Policy Framework Cambodia Financial Technology Development Policy 2023-2028 3.3. Policy Measures Policy measures are designed in line with the above four main goals: (1)-development of policy enablers, (2)-development of digital enablers, (3)- promotion of the use and development of enabling

                    INFO     HTTP Request: POST                                                     ]8;id=11769955;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=11769956;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py#1923\1923]8;;\
                             https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp                
                             "HTTP/1.1 200 OK"                                                                     

QUERY: What are the goals of the e-services for business strategy?
  1. se to the above vision and objective, this strategy has set out four goals as follows: 1. To revise or eliminate services and procedures that are redundant or do not have high value-added, in order to minimize complexity and ensure the ease of applying for business-related services; 2. To build and ...
  2. 10 3. Framework of the Strategy Strategy for the Development of E-Services for Business 2025-2028 3. FRAMEWORK OF THE STRATEGY 3.1. Vision, Objective and Goals To address abovementioned challenges and strengthen intervention measures in line with the government’s policy direction, the “Strategy for ...
  3. STRATEGY FOR THE DEVELOPMENT OF E-SERVICES FOR BUSINESS 2025-2028...



[10/21/26 05:09:25] INFO     HTTP Request: POST                                                     ]8;id=11769961;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py\_client.py]8;;\:]8;id=11769962;file:///Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/httpx2/_client.py#1923\1923]8;;\
                             https://rinabuoy13--workshop-documents-mcp-serve.us-east.modal.run/mcp                
                             "HTTP/1.1 200 OK"                                                                     

QUERY: What are the principles of the IT architecture framework?
  1. 3. INFORMATION TECHNOLOGY ARCHITECTURE FRAMEWORK 16 The Ministry of Economy and Finance Information Technology Architecture Framework 2)- Process: Define the structure of the development, operation, and management of IT systems to change how IT systems work and how they can work together. Furthermor...
  2. 3. INFORMATION TECHNOLOGY ARCHITECTURE FRAMEWORK 23 The Ministry of Economy and Finance Information Technology Architecture Framework (B) Interoperability Framework: refers to the mechanism that allows two or more IT systems to interop with each other to share data or use data from each other. Lack ...
  3. once-only principles, interoperability as an ecosystem principle, and land and expand for efficiency principle. In addition to these three principles, the framework adds six key common principles for implementation, including (1)-Ownership, (2)-Interoperability, (3)-Development Standards, (4)-Open D...



## 8. Connect the server to Claude

**Claude Code.** Run this command in a terminal:

```bash
claude mcp add --transport http workshop-documents <MCP URL>
```

**Claude.ai or Claude Desktop.** Go to *Settings → Connectors → Add custom connector*. Paste the MCP URL.

**Security:** The URL has no authentication. Each person with the URL can search the documents.
Do not use this server for confidential documents. To stop the server, run `modal app stop workshop-documents-mcp`.